# Blocklist Transform Example (Ray)

This notebook demonstrates how to use the blocklist transform with Ray runtime to annotate documents based on blocklisted domains.

## Overview

The blocklist transform identifies documents from blocklisted domains by:
1. Reading domain blocklists from specified files
2. Extracting domains from document URLs
3. Adding an annotation column indicating which documents are from blocklisted domains

This Ray version allows for distributed processing of large datasets.

## Installation

**Note:** These uv pip installs need to be adapted to use the appropriate release level. 

Alternatively, the venv running the jupyter lab could be pre-configured with a requirements file that includes the right release. 

**Example for transform developers working from git clone:**
```bash
make venv
source venv/bin/activate
uv pip install jupyterlab
venv/bin/jupyter lab
```

In [1]:
%%capture
## This is here as a reference only
# Users and application developers must use the right tag for the latest from pypi
%uv pip install "data-prep-toolkit-transforms[ray,blocklist]==1.1.5"

## Import Required Modules

In [2]:
from dpk_blocklist.ray.runtime import Blocklist
from data_processing.utils import GB

## Configure and Run the Transform

### Parameters

The blocklist transform accepts the following key parameters:

* **input_folder**: Path to the input parquet files
* **output_folder**: Path where annotated parquet files will be written
* **blocklist_blocked_domain_list_path**: Path to directory containing domain blocklist files (files matching pattern `domains*`)
* **blocklist_annotation_column_name**: Name of the column to add with blocklist annotations (default: `"blocklisted"`)
* **blocklist_source_url_column_name**: Name of the column containing source URLs (default: `"title"`)

**Ray-specific parameters:**

* **run_locally**: Set to `True` to run Ray locally, `False` to connect to existing Ray cluster
* **num_cpus**: Number of CPUs to allocate per worker (default: 0.8)
* **memory**: Amount of memory to allocate per worker (e.g., `2 * GB`)
* **runtime_num_workers**: Number of Ray workers to spawn (default: determined automatically)

For a full list of parameters, please see the [README](./README.md).

### Example: Basic Blocklist Annotation with Ray

This example shows how to annotate documents using a blocklist of gambling and phishing domains with Ray:

In [3]:
Blocklist(
    input_folder="test-data/input",
    output_folder="output",
    run_locally=True,
    num_cpus=0.8,
    memory=2 * GB,
    runtime_num_workers=2,
    blocklist_blocked_domain_list_path="test-data/domains/arjel",
    blocklist_annotation_column_name="blocklisted",
    blocklist_source_url_column_name="title"
).transform()

17:33:40 INFO - data factory blocklist_ Missing local configuration
17:33:40 INFO - data factory blocklist_ max_files -1, n_sample -1
17:33:40 INFO - data factory blocklist_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
17:33:40 INFO - data factory blocklist_ Data Access:  DataAccessLocal
17:33:40 INFO - pipeline id pipeline_id
17:33:40 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
17:33:40 INFO - number of workers 2 worker options {'num_cpus': 0.8, 'memory': 2147483648, 'max_restarts': -1}
17:33:40 INFO - actor creation delay 0
17:33:40 INFO - job details {'job category': 'preprocessing', 'job name': 'blocklist', 'job type': 'ray', 'job id': 'job_id'}
17:33:40 INFO - data factory data_ max_files -1, n_sample -1
17:33:40 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, file

0

## Verify the Output

The output folder will contain the annotated parquet files. Let's check what was created:

In [4]:
import glob
glob.glob("output/*")

['output\\metadata.json', 'output\\test1.parquet']

## Inspect the Results

Let's read the output parquet file and examine the annotations:

In [5]:
import pyarrow.parquet as pq
import pandas as pd

# Read the output parquet file
output_files = glob.glob("output/*.parquet")
if output_files:
    table = pq.read_table(output_files[0])
    df = table.to_pandas()
    print(f"\nOutput table has {len(df)} rows and {len(df.columns)} columns")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nSample data:")
    print(df.head(10))
    
    # Show blocklist statistics
    blocklisted_count = (df['blocklisted'] != '').sum()
    total_count = len(df)
    print(f"\n=== Blocklist Statistics ===")
    print(f"Total documents: {total_count}")
    print(f"Blocklisted documents: {blocklisted_count}")
    print(f"Clean documents: {total_count - blocklisted_count}")
    print(f"Blocklist rate: {blocklisted_count/total_count*100:.1f}%")
    
    if blocklisted_count > 0:
        print(f"\n=== Blocklisted Domains Found ===")
        print(df[df['blocklisted'] != ''][['title', 'blocklisted']])


Output table has 7 rows and 2 columns

Columns: ['title', 'blocklisted']

Sample data:
                               title    blocklisted
0                      https://poker          poker
1                   https://poker.fr       poker.fr
2              https://poker.foo.bar  poker.foo.bar
3                https://abc.efg.com               
4   http://asdf.qwer.com/welcome.htm               
5  http://aasdf.qwer.com/welcome.htm               
6         https://zxcv.xxx/index.asp               

=== Blocklist Statistics ===
Total documents: 7
Blocklisted documents: 3
Clean documents: 4
Blocklist rate: 42.9%

=== Blocklisted Domains Found ===
                   title    blocklisted
0          https://poker          poker
1       https://poker.fr       poker.fr
2  https://poker.foo.bar  poker.foo.bar


## Check Transform Metadata

The transform also produces a metadata.json file with processing statistics:

In [6]:
import json

metadata_file = "output/metadata.json"
try:
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    print("Transform Metadata:")
    print(json.dumps(metadata, indent=2))
except FileNotFoundError:
    print(f"Metadata file not found at {metadata_file}")

Transform Metadata:
{
  "pipeline": "pipeline_id",
  "job details": {
    "job category": "preprocessing",
    "job name": "blocklist",
    "job type": "ray",
    "job id": "job_id",
    "start_time": "2025-10-16 17:33:57",
    "end_time": "2025-10-16 17:34:09",
    "status": "success"
  },
  "code": {
    "github": "UNDEFINED",
    "build-date": "UNDEFINED",
    "commit_hash": "UNDEFINED",
    "path": "UNDEFINED"
  },
  "job_input_params": {
    "blocked_domain_list_path": "test-data/domains/arjel",
    "annotation_column_name": "blocklisted",
    "source_url_column_name": "title",
    "checkpointing": false,
    "max_files": -1,
    "random_samples": -1,
    "files_to_use": [
      ".parquet"
    ],
    "number of workers": 2,
    "worker options": {
      "num_cpus": 0.8,
      "memory": 2147483648,
      "max_restarts": -1
    },
    "actor creation delay": 0
  },
  "execution_stats": {
    "cpus": 8,
    "gpus": 1,
    "memory": 4.26153030525893,
    "object_store": 2.130765151232

## Example: Using Multiple Workers

For larger datasets, you can increase the number of workers and adjust resources:

In [7]:
# Example with more workers for larger datasets
Blocklist(
    input_folder="large-dataset/input",
    output_folder="large-dataset/output",
    run_locally=True,
    num_cpus=1.0,
    memory=4 * GB,
    runtime_num_workers=8,
    blocklist_blocked_domain_list_path="test-data/domains/gambling",
    blocklist_annotation_column_name="gambling_domain",
    blocklist_source_url_column_name="title"
).transform()

17:35:12 INFO - data factory blocklist_ Missing local configuration
17:35:12 INFO - data factory blocklist_ max_files -1, n_sample -1
17:35:12 INFO - data factory blocklist_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
17:35:12 INFO - data factory blocklist_ Data Access:  DataAccessLocal
17:35:12 INFO - pipeline id pipeline_id
17:35:12 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
17:35:12 INFO - number of workers 8 worker options {'num_cpus': 1.0, 'memory': 4294967296, 'max_restarts': -1}
17:35:12 INFO - actor creation delay 0
17:35:12 INFO - job details {'job category': 'preprocessing', 'job name': 'blocklist', 'job type': 'ray', 'job id': 'job_id'}
17:35:12 INFO - data factory data_ max_files -1, n_sample -1
17:35:12 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, file

0

## Example: Running on Existing Ray Cluster

To use an existing Ray cluster, set `run_locally=False`:

In [ ]:
# Example using existing Ray cluster
import ray
ray.init(address='auto')  # Connect to existing cluster

Blocklist(
    input_folder="s3://my-bucket/input",
    output_folder="s3://my-bucket/output",
    run_locally=False,
    num_cpus=2.0,
    memory=8 * GB,
    runtime_num_workers=16,
    blocklist_blocked_domain_list_path="s3://my-bucket/domains",
    blocklist_annotation_column_name="blocklisted",
    blocklist_source_url_column_name="url"
).transform()

## Integration with Other Transforms

The blocklist transform is often used in combination with the filter transform to remove blocklisted documents:

```python
# Step 1: Annotate with blocklist
from dpk_blocklist.ray.runtime import Blocklist
Blocklist(
    input_folder="input",
    output_folder="annotated",
    run_locally=True,
    blocklist_blocked_domain_list_path="domains"
).transform()

# Step 2: Filter out blocklisted documents
from dpk_filter.ray.runtime import Filter
Filter(
    input_folder="annotated",
    output_folder="filtered",
    run_locally=True,
    filter_criteria_list=["blocklisted = ''"],  # Keep only non-blocklisted
    filter_columns_to_drop=["blocklisted"]       # Remove the annotation column
).transform()
```

## Summary

This notebook demonstrated:
- Installing and importing the blocklist transform with Ray support
- Running the transform with Ray runtime parameters
- Configuring worker resources and parallelism
- Inspecting the annotated output
- Using Ray locally vs. on existing clusters
- Integrating with other transforms

For more information, see:
- [Blocklist Transform README](./README.md)
- [Data Prep Kit Documentation](https://github.com/data-prep-kit/data-prep-kit)
- [Ray Documentation](https://docs.ray.io/)
- [Transform Project Conventions](../../README.md)